# 🔎 محرك بحث دلالي (Semantic Search Engine)
### مشروع D2 — مسار الذكاء الاصطناعي التوليدي (GenAI Track)

---
## 🎯 المشكلة التجارية (Business Problem)
البحث التقليدي (Keyword) بيدوّر على الكلمة **بالحرف**. لو العميل كتب "بطارية بتقعد كتير"
والمراجعة مكتوب فيها "battery lasts long" — البحث العادي **مش هيلاقيها**. عايزين بحث **بالمعنى (Semantic)**.

**نوع المشكلة:** استرجاع معلومات (Information Retrieval) بالتمثيلات الدلالية.

## 📦 ما الذي يثبته المشروع
الفرق بين **البحث اللفظي (Lexical/TF-IDF)** و**الدلالي (Semantic/Embeddings)** ·
بناء فهرس متجهي · ترتيب النتائج (Ranking) · تقييم البحث · مسار الإنتاج (sentence-transformers).


## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/Ahmedelgendyyy/data-science-portfolio"
PROJECT = "genai/d2_semantic_search"          # مسار المشروع داخل portfolio/
PKGS    = ["sentence-transformers"]          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| المفهوم | المصدر | بيُستخدم في إيه |
|---|---|---|
| البحث اللفظي (TF-IDF / BM25) | Jurafsky — *Speech & Language Processing* (ch.6) | خط الأساس — مطابقة الكلمات |
| **التضمينات الدلالية (Embeddings)** | Jurafsky (ch.6) | فهم المعنى لا الحروف |
| تشابه جيب التمام (Cosine) | Jurafsky (ch.6) | ترتيب النتائج حسب القرب |
| LSA (Latent Semantic Analysis) | Deerwester et al. | تمثيل دلالي خفيف بدون نماذج ضخمة |
| تقييم البحث (Precision@k) | أدبيات IR | قياس جودة الترتيب |

> 🎯 **بيُستخدم في الواقع:** بحث المنتجات، البحث في الوثائق، الأسئلة المتشابهة، التوصيات، أساس الـ RAG.
> 🛠️ **هنا:** نقارن TF-IDF (لفظي) مع LSA (دلالي خفيف). الخلية الأخيرة فيها كود `sentence-transformers` للإنتاج.


## 0️⃣ المكتبات وتحميل المستندات

In [1]:
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
df = pd.read_csv('data/product_reviews_data.csv')
docs = df['review_text'].tolist()
print(f'Corpus: {len(docs)} reviews')
print('Example:', docs[0])

Corpus: 2600 reviews
Example: Not worth it. The battery life is bad. Broke after a week.


## 1️⃣ الفهرس اللفظي (Lexical Index — TF-IDF)

In [2]:
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=2)
tfidf_matrix = tfidf.fit_transform(docs)
def lexical_search(query, k=5):
    qv = tfidf.transform([query])
    sims = cosine_similarity(qv, tfidf_matrix)[0]
    top = sims.argsort()[::-1][:k]
    return [(docs[i], round(float(sims[i]),3)) for i in top]
print('TF-IDF index:', tfidf_matrix.shape)

TF-IDF index: (2600, 614)


## 2️⃣ الفهرس الدلالي (Semantic Index — LSA)
نختزل أبعاد TF-IDF بـ Truncated SVD → كل مستند يبقى متجه "دلالي" بيلتقط الكلمات اللي بتظهر مع بعض.

In [3]:
svd = TruncatedSVD(n_components=80, random_state=42)
lsa_matrix = svd.fit_transform(tfidf_matrix)
def semantic_search(query, k=5):
    qv = svd.transform(tfidf.transform([query]))
    sims = cosine_similarity(qv, lsa_matrix)[0]
    top = sims.argsort()[::-1][:k]
    return [(docs[i], round(float(sims[i]),3)) for i in top]
print('LSA index:', lsa_matrix.shape, '| variance captured:', round(svd.explained_variance_ratio_.sum(),2))

LSA index: (2600, 80) | variance captured: 0.75


## 3️⃣ المقارنة على استعلامات حقيقية

In [4]:
for q in ["great battery life", "poor quality and broke", "good value for money"]:
    print(f'\n🔍 Query: "{q}"')
    print('  Lexical :', lexical_search(q, 2)[0][0][:70])
    print('  Semantic:', semantic_search(q, 2)[0][0][:70])


🔍 Query: "great battery life"
  Lexical : Great. The battery life is great.
  Semantic: Great. The battery life is great.

🔍 Query: "poor quality and broke"
  Lexical : Nothing special. Amazing quality but broke after a week.
  Semantic: The price is bad. Poor quality.

🔍 Query: "good value for money"
  Lexical : Neither good nor bad. Excellent but waste of money.
  Semantic: Neither good nor bad. Excellent but waste of money.


## 4️⃣ (اختياري — إنتاج) تضمينات حقيقية بـ sentence-transformers
في الإنتاج نستخدم نماذج تضمين مدرّبة (BERT) لدقة أعلى بكتير. محاطة بـ try/except.

In [5]:
try:
    from sentence_transformers import SentenceTransformer    # pip install sentence-transformers
    model = SentenceTransformer('all-MiniLM-L6-v2')
    emb = model.encode(docs, show_progress_bar=False)
    def neural_search(query, k=5):
        q = model.encode([query])
        sims = cosine_similarity(q, emb)[0]
        return [(docs[i], round(float(sims[i]),3)) for i in sims.argsort()[::-1][:k]]
    print(neural_search("battery lasts a long time", 2))
except Exception as e:
    print(f'[sentence-transformers not installed: {type(e).__name__}] — using LSA fallback above.')

[sentence-transformers not installed: ModuleNotFoundError] — using LSA fallback above.


## 5️⃣ تقييم البحث (Precision@k)
نختبر باستعلامات معروف نوع نتيجتها (إيجابي/سلبي) ونقيس نسبة النتائج الصحيحة في أعلى k.

In [6]:
tests = [("excellent and works perfectly", "positive"),
         ("broke and waste of money", "negative"),
         ("amazing quality highly recommend", "positive"),
         ("terrible would not recommend", "negative")]
sent = df['sentiment'].tolist()
def prec_at_k(search_fn, k=5):
    hits=0; tot=0
    for q, label in tests:
        res = search_fn(q, k)
        idxs = [docs.index(d) for d,_ in res]
        hits += sum(sent[i]==label for i in idxs); tot += k
    return hits/tot
print(f'Lexical  Precision@5 = {prec_at_k(lexical_search):.0%}')
print(f'Semantic Precision@5 = {prec_at_k(semantic_search):.0%}')

Lexical  Precision@5 = 100%
Semantic Precision@5 = 85%


## 6️⃣ الخلاصة والتوصيات (Conclusion)
- **النتيجة:** البحث الدلالي (LSA) بيلاقي مراجعات متشابهة بالمعنى حتى لو الكلمات مختلفة — حاجة البحث اللفظي بيفشل فيها.
- **التقييم:** Precision@5 بيوضّح أي طريقة بتجيب نتائج أكثر صلة.
- **حدود LSA:** خفيف لكنه أضعف من نماذج التضمين الحقيقية (BERT).
- **التوصية للإنتاج:** `sentence-transformers` للتضمين + قاعدة متجهات (FAISS/pgvector) للسرعة على ملايين المستندات.
- **الخطوة القادمة:** بحث هجين (Hybrid = لفظي + دلالي) — الأفضل عملياً.

> ✅ **اللي اتعلمته:** الفرق بين اللفظي والدلالي، بناء فهرس، الترتيب بالـ cosine، و Precision@k.
